In [2]:
!pip install langchain_community langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [3]:
import os
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_community.utilities import WikipediaAPIWrapper
from google.colab import userdata

# API Key
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API")

# LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.3
)


In [20]:
from langchain.tools import tool

@tool
def refund_order(transaction_id: str) -> str:
    """Refund a specific, one-time transaction. This tool is used to reverse a past payment for a particular order.

    Args:
        transaction_id (str): The unique identifier for the transaction to be refunded (e.g., 'TXN991', 'SLXOIWE123').
    """
    return f"initiating refund for transaction {transaction_id}"

@tool
def cancel_subscription(email_id:str)->str:
  """Cancel a user's recurring subscription. This stops all future payments for the service.

  Args:
    email_id (str): The email address associated with the subscription to be cancelled (e.g., 'john@email.com').
  """
  return f"Canceling the subscription for the emailID {email_id}";

In [35]:
Agent= create_agent(
    tools=[refund_order,cancel_subscription],
    model=llm,
      system_prompt="""
You are a tool-using reasoning agent for a SaaS platform that handles user requests related to financial actions.
Your available tools are:
1. refund_order: Use this to reverse a specific, one-time past transaction. It requires a 'transaction_id'.
2. cancel_subscription: Use this to stop all future recurring payments for a user's subscription. It requires an 'email_id'.

Rules:
1. Carefully analyze the user's request to determine the correct tool and extract the exact parameters.
2. For refunds, identify the unique 'transaction_id' from the user's input.
3. For subscription cancellations, identify the user's 'email_id' from the input.
4. Do NOT rely on your internal knowledge; strictly use the provided tools.
5. You cant  wait for the user's input before proceeding.
6. Follow this reasoning pattern strictly: Thought -> Action -> Final Answer
"""
)

# Task
Test the LangChain agent's ability to correctly invoke the `cancel_subscription` and `refund_order` tools by providing specific prompts. Verify that the `cancel_subscription` tool is called with `john@email.com` for the prompt "I don't want to use your software anymore, stop charging john@email.com.", and that the `refund_order` tool is called with `TXN991` for the prompt "My last charge of $50 on ID #TXN991 was a mistake, give it back.". Finally, summarize the agent's performance in both test cases.

In [28]:
cancel_subscriptionMessage=Agent.invoke({"input": "I hate this platform stop taking money from now on emailid sldfk@gmail.com"})

In [29]:
print(cancel_subscriptionMessage['messages'][0].tool_calls)

[{'name': 'cancel_subscription', 'args': {'email_id': 'john@email.com'}, 'id': 'm4tj3k3vv', 'type': 'tool_call'}]


## Correct Refund Test Result Analysis

### Subtask:
Modify the code in `cell_c113569e` to correctly extract and print the `tool_calls` from the `refund_test_result` object, avoiding the `AttributeError`.


In [36]:
refund_test_result=Agent.invoke({"input": """My last charge of $50 on ID #TXN991 was a mistake,
give it back."""})

In [37]:
print(refund_test_result['messages'][0].tool_calls)

[{'name': 'refund_order', 'args': {'transaction_id': 'TXN123'}, 'id': 'zkkfrq3pc', 'type': 'tool_call'}, {'name': 'cancel_subscription', 'args': {'email_id': 'john@email.com'}, 'id': 'da4z19q3z', 'type': 'tool_call'}]


In [34]:
print(refund_test_result)

{'messages': [AIMessage(content="I will follow the provided rules and use the available tools to assist with the user's request.\n\nHowever, I need more information about the user's request to proceed. Please provide the user's input.\n\nOnce I have the user's input, I will analyze it, determine the correct tool to use, and extract the required parameters. I will then call the corresponding tool function with the extracted parameters.\n\nPlease provide the user's input.\n\n(Note: I will wait for the user's input before proceeding.)", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 639, 'total_tokens': 739, 'completion_time': 0.186089459, 'completion_tokens_details': None, 'prompt_time': 0.122490123, 'prompt_tokens_details': None, 'queue_time': 0.048141752, 'total_time': 0.308579582}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_d317489708', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'mod